# The stratum server: one reuse cache for many runs

A stratum server is one long-lived Python process that runs pipelines for its
clients, and all of them share its reuse cache. An intermediate result one run
computed (a table read, the features derived from it, a fitted model) is taken from
the cache by the next run that needs the same value.

This notebook is the client: it builds skrub DAGs and sends them to the server,
which grid-searches or evaluates them. The server runs in a terminal.

## 1. Start the server in a terminal

From the repository root, with the `server` extra installed
(`pip install -e ".[server]"`). In PyCharm, its terminal already uses the project's
interpreter; the notebook's kernel must use the same one:

```bash
python -m stratum serve
```

It prints `stratum server listening on 127.0.0.1:<port> (reuse cache ..., 0 entries)`
and keeps running. It writes its address and a token under `~/.stratum/server/`,
where the client finds them. The cache lives in `.stratum/reuse_cache` under the
directory the server started in; `--cache-dir DIR` puts it elsewhere.

## 2. Connect

In [7]:
from pathlib import Path

from stratum import StratumClient

client = StratumClient()  # the local server, found under ~/.stratum/server
client.health()

{'ok': True, 'version': '0.0.0.dev2', 'jobs': 0, 'queued': 0}

## 3. The data

A synthetic sales table of a million rows, written next to this notebook. The DAG
names it by its absolute path, so the server reads the same file.

In [8]:
import numpy as np
import pandas as pd

# examples/server/, whether the kernel starts there or in the project root.
HERE = next(d for d in (Path.cwd(), Path.cwd() / "examples" / "server")
            if (d / "server_client.ipynb").exists())
DATA = HERE / "sales.csv"


def make_sales(path, n=1_000_000):
    rng = np.random.default_rng(0)
    store = rng.integers(0, 200, n)
    price = rng.gamma(2.0, 10.0, n)
    quantity = rng.poisson(3, n) + 1
    pd.DataFrame({
        "store": store,
        "price": price,
        "quantity": quantity,
        "discount": rng.random(n) * 0.3,
        "revenue": price * quantity * (1 + store / 400) + rng.normal(0, 5, n),
    }).to_csv(path, index=False)


if not DATA.exists():
    make_sales(DATA)
pd.read_csv(DATA, nrows=5)

,store,price,quantity,discount,revenue
0,170,24.388495,4,0.019717,132.769349
1,127,24.225889,5,0.091330,157.649018
2,102,18.742012,1,0.070015,21.618837
3,53,38.350350,5,0.254190,222.498310
4,61,32.838026,7,0.178036,267.643803


## 4. The pipeline

Built here, lazily: nothing runs in this kernel. It reads the sales table, derives
features and grid-searches the alpha of a Ridge model with 3-fold cross-validation.
The client pickles the DAG; library functions such as `pd.read_csv` and `np.log1p`
travel by name, and the server uses its own pandas and numpy.

Only a file source gets a cache key: the server names it by its path and file
metadata, never by reading it, so an edited file is a new value. Every step here is
one stratum understands, so each gets a key derived from the file's.

In [9]:
import skrub
from sklearn.linear_model import Ridge

import stratum as st


def sales_table():
    sales = st.as_data_op(str(DATA)).skb.apply_func(pd.read_csv)
    return sales.assign(net_price=sales["price"] * (1 - sales["discount"]),
                        log_quantity=sales["quantity"].skb.apply_func(np.log1p))


def pipeline(alphas):
    with skrub.config_context(eager_data_ops=False):
        sales = sales_table()
        X = sales[["store", "price", "discount", "net_price", "log_quantity"]].skb.mark_as_X()
        y = sales["revenue"].skb.mark_as_y()
        return X.skb.apply(Ridge(alpha=skrub.choose_from(alphas, name="alpha")), y=y)

`search` sends the pipeline to the server and returns its results table. It also
reports what the run took from the cache (hits) and what it left in it (admitted),
from the server's statistics before and after.

In [10]:
import time


def search(*alphas):
    before = client.stats()["stats"]
    start = time.perf_counter()
    results = client.grid_search(pipeline(list(alphas)), cv=3, scoring="r2")
    after = client.stats()["stats"]
    print(f"{time.perf_counter() - start:.2f}s: "
          f"{after['planned_hits'] - before['planned_hits']} values from the cache, "
          f"{after['planned_admissions'] - before['planned_admissions']} added to it")
    return results

## 5. A first search fills the cache

In [11]:
search(1, 10_000)

2.02s: 0 values from the cache, 16 added to it


id,scores
str,f64
"""alpha:1""",0.856175
"""alpha:10000""",0.853822


## 6. Other alphas: the features come from the cache

The table and its features do not depend on alpha, so only the models are fitted
again.

In [12]:
search(100_000, 1_000_000)

2.30s: 2 values from the cache, 12 added to it


id,scores
str,f64
"""alpha:100000""",0.792968
"""alpha:1000000""",0.669002


## 7. The same alphas: the fitted models come from the cache too

Each fold's fitted Ridge is cached beside its output, so nothing is fitted again.

In [ ]:
search(100_000, 1_000_000)

## 8. A query: evaluate

A DAG without X and y is a query: `evaluate` returns the table it computes. It reads
the same sales table as the searches, so the server takes it from its cache.

`share_of_total` is defined in this notebook, so the server has no module to import
it from: the client sends it by value, with the DAG (as it would a lambda, or a
helper module next to the notebook). The server cannot see into such a function, so
it gives its output no cache key, nor anything computed from it.

In [ ]:
def share_of_total(table):
    return table.assign(share=table["revenue"] / table["revenue"].sum())


with skrub.config_context(eager_data_ops=False):
    sales = sales_table()
    per_store = (sales.groupby("store", as_index=False)
                 .agg(revenue=("revenue", "sum"), orders=("quantity", "size"))
                 .sort_values("revenue", ascending=False)
                 .skb.apply_func(share_of_total))

before = client.stats()["stats"]["planned_hits"]
top = client.evaluate(per_store)
print(client.stats()["stats"]["planned_hits"] - before, "values from the cache")
top.head()

## 9. The cache

In [ ]:
{key: value for key, value in client.stats().items() if key != "stats"}

In [ ]:
client.stats()["stats"]

## 10. The cache outlives the server

`shutdown` waits for queued jobs, writes the cache's memory tier to disk and stops the
server. Start it again in the terminal (`python -m stratum serve`), then reconnect:
the next search reads the cached values from disk (`disk_hits`).

In [ ]:
client.shutdown()

In [ ]:
client = StratumClient()  # after restarting the server
search(100_000, 1_000_000)
client.stats()["stats"]["disk_hits"]

## 11. Clean up

`clear` deletes every cache entry. The terminal commands do the same:
`python -m stratum stats`, `python -m stratum clear` and `python -m stratum shutdown`.
The server also runs whole scripts: `python -m stratum submit script.py [args]`, or
`client.submit("script.py", [...])`.

In [ ]:
client.clear()

In [ ]:
client.shutdown()
client.close()